#Synthetic Data Genaration for Resource Allocation

In [ ]:
import pandas as pd
import numpy as np
import random

# ==========================================
# 1. LOAD DATASETS
# ==========================================
print("📂 Loading datasets...")

# Load Crime Data
try:
    df_crime = pd.read_csv('CrimeData_Final.csv')
    print(f"✅ Loaded Crime Data: {len(df_crime)} records")
except FileNotFoundError:
    print("❌ Error: 'CrimeData_Final.csv' not found. Please upload it.")

# Load Mapping Data (GN to DS)
try:
    df_map = pd.read_csv('kandy_ds_to_gn.csv')
    print(f"✅ Loaded GN Map: {len(df_map)} GN Divisions found")
except FileNotFoundError:
    print("❌ Error: 'kandy_ds_to_gn.csv' not found. Please upload it.")

# ==========================================
# 2. CREATE REALISTIC ZONE MAPPING
# ==========================================
# We use 'admin3Name_en' (DS Division) as the Police/Patrol Zone
# We use 'admin4Name_en' (GN Division) to link crime locations to zones

# Create a dictionary: { "Welata": "Gangawata Korale", ... }
# Normalize text (strip whitespace, lowercase) to maximize matches
df_map['admin4Name_clean'] = df_map['admin4Name_en'].str.strip().str.lower()
df_map['admin3Name_clean'] = df_map['admin3Name_en'].str.strip()

gn_to_ds_map = pd.Series(
    df_map.admin3Name_clean.values,
    index=df_map.admin4Name_clean
).to_dict()

# Get list of unique Patrol Zones (The 20ish DS Divisions in Kandy)
patrol_zones = df_map['admin3Name_clean'].unique()
print(f"ℹ️  Identified {len(patrol_zones)} Real Patrol Zones (DS Divisions):")
print(patrol_zones[:5]) # Show first 5

# ==========================================
# 3. GENERATE: POLICE ROSTER (Supply Side)
# ==========================================
# Generating officer availability for REAL DS Divisions

shifts = ['Morning (06:00-14:00)', 'Evening (14:00-22:00)', 'Night (22:00-06:00)']
dates = pd.date_range(start='2025-01-01', periods=30)
roster_data = []

print("\n👮 Generating Synthetic Police Roster...")

for date in dates:
    for zone in patrol_zones:
        for shift in shifts:
            # Logic: Urban areas (like Gangawata Korale) have more officers than rural ones
            # We can use the 'DS_population' from your new file if available, or simple heuristics

            is_urban = zone in ['Gangawata Korale', 'Kundasale', 'Yatinuwara']

            if is_urban:
                if 'Evening' in shift: base_officers = np.random.randint(30, 45)
                elif 'Morning' in shift: base_officers = np.random.randint(25, 35)
                else: base_officers = np.random.randint(15, 25)
            else:
                if 'Evening' in shift: base_officers = np.random.randint(10, 15)
                elif 'Morning' in shift: base_officers = np.random.randint(8, 12)
                else: base_officers = np.random.randint(4, 8)

            roster_data.append({
                'Date': date.strftime('%Y-%m-%d'),
                'Patrol_Zone_DS': zone,
                'Shift_ID': shift,
                'Total_Officers_Available': base_officers,
                'Target_Response_Time_Mins': 15 if is_urban else 25,
                # Actual response is usually slower than target
                'Avg_Actual_Response_Time_Mins': (15 if is_urban else 25) + np.random.randint(0, 15)
            })

df_roster = pd.DataFrame(roster_data)
df_roster.to_csv('synthetic_police_roster_real_zones.csv', index=False)
print("✅ Output Saved: 'synthetic_police_roster_real_zones.csv'")

# ==========================================
# 4. GENERATE: CRIME SEVERITY (Training Data)
# ==========================================
print("\n⚖️  Generating Crime Severity Training Data...")

df_crime_augmented = df_crime.copy()

# Function to find the DS Division for a Crime Record
def get_ds_division(gn_name):
    clean_name = str(gn_name).strip().lower()
    return gn_to_ds_map.get(clean_name, "Unknown_Zone") # Fallback if name doesn't match

df_crime_augmented['Patrol_Zone_DS'] = df_crime_augmented['gn_division'].apply(get_ds_division)

# Severity Weights
severity_weights = {
    'theft': 15, 'vehicle theft': 25, 'burglary': 40,
    'drugs': 55, 'robbery': 65, 'stabbing': 85, 'homicide': 100
}

def calculate_harm_score(row):
    # 1. Crime Type Base Score
    base = severity_weights.get(row.get('crime'), 30)

    # 2. Zone Factor (Urban centers might have 'High Profile' weighting in policy)
    zone_weight = 5 if row['Patrol_Zone_DS'] == 'Gangawata Korale' else 0

    # 3. Time Factor (Night crimes are riskier)
    try:
        hour = int(str(row.get('time')).split(':')[0])
        time_weight = 10 if (hour >= 22 or hour <= 4) else 0
    except:
        time_weight = 0

    # 4. Random Noise (to make it learnable)
    noise = np.random.randint(-5, 5)

    return min(100, max(0, base + zone_weight + time_weight + noise))

df_crime_augmented['Harm_Score'] = df_crime_augmented.apply(calculate_harm_score, axis=1)

# Save
model_cols = ['crime', 'gn_division', 'Patrol_Zone_DS', 'time', 'victim_age', 'Harm_Score']
df_crime_augmented[model_cols].to_csv('synthetic_crime_severity_real_zones.csv', index=False)
print("✅ Output Saved: 'synthetic_crime_severity_real_zones.csv'")

# ==========================================
# 5. GENERATE: PATROL HISTORY (Optimization Curve)
# ==========================================
# This remains the same as before (generic math curve),
# but we can now tag it with real zones if needed.
print("\n📈 Generating Patrol Effectiveness History...")

history_data = []
for zone in patrol_zones: # Generate history for each real zone
    for _ in range(50): # 50 historical shifts per zone
        officers = np.random.randint(1, 40)

        # Diminishing returns formula
        # Efficiency might be lower in large rural areas (spread out)
        is_large_rural = zone not in ['Gangawata Korale']
        decay_rate = 0.08 if is_large_rural else 0.12  # Urban saturates faster

        effectiveness = 100 * (1 - np.exp(-decay_rate * officers))
        crime_prevented_score = effectiveness + np.random.normal(0, 3)

        history_data.append({
            'Patrol_Zone_DS': zone,
            'Shift_ID': random.choice(shifts),
            'Officers_Deployed': officers,
            'Crime_Prevention_Score': round(max(0, crime_prevented_score), 2)
        })

df_history = pd.DataFrame(history_data)
df_history.to_csv('synthetic_patrol_history_real_zones.csv', index=False)
print("✅ Output Saved: 'synthetic_patrol_history_real_zones.csv'")

print("\n🎉 Done! You now have 3 files mapped to Kandy's real administrative divisions.")

📂 Loading datasets...
✅ Loaded Crime Data: 1608 records
✅ Loaded GN Map: 1187 GN Divisions found
ℹ️  Identified 20 Real Patrol Zones (DS Divisions):
['Akurana' 'Deltota' 'Doluwa' 'Ganga Ihala Korale' 'Gangawata Korale']

👮 Generating Synthetic Police Roster...
✅ Output Saved: 'synthetic_police_roster_real_zones.csv'

⚖️  Generating Crime Severity Training Data...
✅ Output Saved: 'synthetic_crime_severity_real_zones.csv'

📈 Generating Patrol Effectiveness History...
✅ Output Saved: 'synthetic_patrol_history_real_zones.csv'

🎉 Done! You now have 3 files mapped to Kandy's real administrative divisions.
